# 07i — Threshold sensitivity + BCL-primary semantic interpretation

This stage audits whether conservative heuristic thresholds could explain low OFFICE / stable-secondary semantic support.

Two methodological rules are frozen before looking at results:

1. BCL POI 2008 is the primary historical POI source for the 2007–2009 GeoLife-heavy cohort. Historical OSM is support-only: missing/negative OSM is not treated as contradiction.
2. Sensitivity grids are predeclared. This notebook does not choose a new production threshold because it improves BCL agreement.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))

def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)

import numpy as np
from geolife.model import HomeOfficeConfig, build_semantic_locations

spec=importlib.util.spec_from_file_location('s07i',REPO_DIR/'analysis'/'07i_threshold_sensitivity.py')
assert spec is not None and spec.loader is not None
s07i=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07i
spec.loader.exec_module(s07i)

print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print(s07i.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07i_threshold_sensitivity'; C.mkdir(parents=True,exist_ok=True)

def find_stay_cache():
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
        REPO_DIR/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for p in candidates:
        if p.exists(): return p
    matches=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if matches: return matches[0]
    raise FileNotFoundError('Cannot find stays_baseline_v1.pkl')

STAYS=find_stay_cache()
C05=V/'cache'/'cp2_v2'/'05_home_office_reliability_validation'
C05B=V/'cache'/'cp2_v2'/'05b_home_consensus_adaptive_work'
C07C=V/'cache'/'cp2_v2'/'07c_historical_source_alignment'
C07G=V/'cache'/'cp2_v2'/'07g_bcl_poi_2008_alignment'
C07H=V/'cache'/'cp2_v2'/'07h_evidence_triangulation'

paths={
 'assignments':C05/'assignments_private.pkl',
 'home':C05B/'home_consensus_private.pkl',
 'baseline_patterns':C05B/'adaptive_work_patterns_42_private.pkl',
 'anchors':C07C/'anchor_observation_dates_private.pkl',
 'classified_pois':C07G/'bcl_neighborhood_pois_classified_private.pkl',
 'profiles':V/'cache'/'cp2_v2'/'07b_factorized_work_profiles'/'factorized_work_profiles_private.pkl',
 'triangulation':C07H/'stable_secondary_triangulation_private.pkl',
}
missing=[str(p) for p in [STAYS,*paths.values()] if not p.exists()]
if missing: raise FileNotFoundError('Missing upstream artifacts:\n'+'\n'.join(missing))

stays=pd.read_pickle(STAYS)
assignments=pd.read_pickle(paths['assignments'])
home_evidence=pd.read_pickle(paths['home'])
baseline_saved=pd.read_pickle(paths['baseline_patterns'])
anchor_dates=pd.read_pickle(paths['anchors'])
classified_pois=pd.read_pickle(paths['classified_pois'])
profiles=pd.read_pickle(paths['profiles'])
triangulation=pd.read_pickle(paths['triangulation'])

semantic_stays,locations=build_semantic_locations(
    stays,
    config=HomeOfficeConfig(clustering_method='complete_link',location_max_diameter_m=200.0),
)
print('semantic users:',semantic_stays['user_id'].nunique(),'locations:',len(locations))
print('saved baseline stable users:',int(baseline_saved['window_pattern'].eq('stable_secondary_anchor').sum()))


In [ ]:
# BCL primary metrics: reuse Stage-07g extraction up to its frozen 150m padding.
eligible=s07i.STAGE07G.bcl_eligible_anchors(anchor_dates)
matches150,bcl_metrics=s07i.build_bcl_metrics_to_150m(eligible,classified_pois)
print('BCL eligible anchors:',len(eligible),'users:',eligible['user_id'].nunique())
print('anchor-POI pairs <=150m:',len(matches150))

# Baseline reproduction gate before any sensitivity interpretation.
base_cfg=s07i.MobilitySensitivityConfig(name='baseline_recomputed')
base_windows,base_patterns=s07i.run_mobility_config(semantic_stays,home_evidence,assignments,base_cfg)
assert int(base_patterns['window_pattern'].eq('stable_secondary_anchor').sum())==9

saved_stable=(baseline_saved.loc[baseline_saved['window_pattern'].eq('stable_secondary_anchor'),['user_id','dominant_location_id']]
              .assign(user_id=lambda x:x['user_id'].astype(str))
              .sort_values('user_id').reset_index(drop=True))
recomputed_stable=(base_patterns.loc[base_patterns['window_pattern'].eq('stable_secondary_anchor'),['user_id','dominant_location_id']]
                   .assign(user_id=lambda x:x['user_id'].astype(str))
                   .sort_values('user_id').reset_index(drop=True))
pd.testing.assert_frame_equal(saved_stable,recomputed_stable,check_dtype=False)
print('baseline 05b reproduction: PASS (9 users, candidate IDs exact)')


In [ ]:
# 1) Production fixed-window OFFICE gate sensitivity (27 predeclared configs).
office_grid,office_details=s07i.run_office_gate_sensitivity(semantic_stays)
office_grid=s07i.annotate_office_grid_with_bcl(office_grid,office_details,bcl_metrics,threshold_m=100.0)
baseline_office=office_grid.loc[office_grid['is_frozen_baseline']]
assert len(baseline_office)==1
assert int(baseline_office.iloc[0]['emitted_users'])==16
print('frozen production OFFICE reproduction: PASS (16 users)')
display(office_grid.sort_values(['office_min_dates','office_min_share','office_min_margin']).reset_index(drop=True))


In [ ]:
# 2) Adaptive stable-secondary sensitivity: one-at-a-time + joint profiles.
mobility_rows=[]
bcl_rows=[]
bcl_category_rows=[]
pattern_private=[]

for cfg in s07i.all_mobility_configs():
    windows,patterns=s07i.run_mobility_config(semantic_stays,home_evidence,assignments,cfg)
    mobility_rows.append(s07i.summarize_mobility_config(patterns,config=cfg,baseline_patterns=base_patterns))
    summary,categories=s07i.summarize_bcl_for_patterns(
        bcl_metrics,patterns,profiles,config_name=cfg.name,family=cfg.family,radii_m=s07i.BCL_RADII_M
    )
    bcl_rows.append(summary); bcl_category_rows.append(categories)
    keep=patterns.copy(); keep['config_name']=cfg.name; keep['family']=cfg.family
    pattern_private.append(keep)

mobility_summary=pd.DataFrame(mobility_rows)
bcl_sensitivity=pd.concat(bcl_rows,ignore_index=True)
bcl_category_sensitivity=pd.concat(bcl_category_rows,ignore_index=True)
mobility_patterns_private=pd.concat(pattern_private,ignore_index=True)

print('ONE-AT-A-TIME MOBILITY SENSITIVITY')
display(mobility_summary.loc[mobility_summary['family'].eq('oat')])
print('JOINT MOBILITY PROFILES')
display(mobility_summary.loc[mobility_summary['family'].eq('profile')])


In [ ]:
# 3) POI radius sensitivity and BCL-primary / OSM-support-only reinterpretation.
baseline_bcl=bcl_sensitivity.loc[bcl_sensitivity['config_name'].eq('baseline')].copy()
print('BASELINE STABLE-SECONDARY — BCL RADIUS SENSITIVITY')
display(baseline_bcl)

baseline_categories=bcl_category_sensitivity.loc[
    bcl_category_sensitivity['config_name'].eq('baseline')
].copy()
print('BASELINE CANDIDATE CATEGORY COUNTS — BCL')
display(baseline_categories)

bcl_primary_osm_support=s07i.bcl_primary_osm_support_summary(triangulation,threshold_m=100.0)
print('BCL PRIMARY / OSM SUPPORT-ONLY SNAPSHOT')
display(bcl_primary_osm_support)


In [ ]:
# Compact comparison views: do relaxed thresholds add stable users, and does BCL evidence improve consistently?
profile_names=['relaxed','mildly_relaxed','baseline_profile','strict']
profile_mobility=mobility_summary.loc[mobility_summary['config_name'].isin(profile_names)].copy()
profile_bcl=bcl_sensitivity.loc[
    bcl_sensitivity['config_name'].isin(profile_names)
    & bcl_sensitivity['threshold_m'].isin([100.0,150.0])
].copy()
print('JOINT PROFILE — MOBILITY COUNTS')
display(profile_mobility)
print('JOINT PROFILE — BCL CANDIDATE VS PEERS @100/150m')
display(profile_bcl)


In [ ]:
# Save private user-level sensitivity and aggregate tables.
office_details.to_pickle(C/'office_gate_emitted_details_private.pkl')
mobility_patterns_private.to_pickle(C/'mobility_patterns_sensitivity_private.pkl')
matches150.to_pickle(C/'bcl_anchor_poi_matches_150m_private.pkl')
bcl_metrics.to_pickle(C/'bcl_anchor_metrics_150m_private.pkl')

office_grid.to_csv(C/'office_gate_sensitivity.csv',index=False)
mobility_summary.to_csv(C/'mobility_gate_sensitivity.csv',index=False)
bcl_sensitivity.to_csv(C/'bcl_radius_and_mobility_sensitivity.csv',index=False)
bcl_category_sensitivity.to_csv(C/'bcl_candidate_category_sensitivity.csv',index=False)
bcl_primary_osm_support.to_csv(C/'bcl_primary_osm_support_snapshot.csv',index=False)
print('saved:',C)


## Interpretation guardrails

- OFFICE gate sensitivity measures coverage/identity stability only; it does not prove relaxed emissions are correct.
- BCL is the primary historical POI source. OSM may positively support a BCL-favoring candidate, but OSM absence/negative direction is not counted as contradiction.
- Radius sensitivity is limited to 150 m because Stage 07g extracted POIs with 150 m retrieval padding. Testing 200 m would require a new extraction and is intentionally not done here.
- No threshold is selected in this notebook based on better BCL agreement. Any production change requires a separate predeclared decision rule or stronger ground truth.
